# ONNX Runtime Provider Test

A minimal speed test example for the `CPU`, `CUDA`, and `TensorRT` providers.


In [ ]:
# Check the current CUDA version to understand which TensorRT package to install
!nvidia-smi


In [ ]:
# Install once in your notebook environment.
# !pip install "onnxruntime-gpu" tensorrt

## Smoke Test


In [4]:
import numpy as np
import onnxruntime as ort

model_path = "model/model.onnx"
input_data = np.random.rand(1, 3, 224, 224).astype(np.float32)
sess = ort.InferenceSession(model_path,
                            providers=["TensorrtExecutionProvider"])
inp_name = sess.get_inputs()[0].name
feed = {inp_name: input_data}
out = sess.run(None, feed)

print(f"providers={sess.get_providers()}  output={out[0].shape}")

providers=['TensorrtExecutionProvider', 'CPUExecutionProvider']  output=(1, 1000)


Multiple - run helper 

In [5]:
import time
import numpy as np


def bench(name, providers, runs=20, warmup=5, batch =1):
    input_data = np.random.rand(batch, 3, 224, 224).astype(np.float32)
    sess = ort.InferenceSession(model_path, providers=providers)
    inp_name = sess.get_inputs()[0].name
    feed = {inp_name: input_data}

    for _ in range(warmup):
        sess.run(None, feed)

    t0 = time.perf_counter()
    for _ in range(runs):
        out = sess.run(None, feed)
    ms = (time.perf_counter() - t0) * 1000 / runs

    print(f"{name:14} {ms:8.2f} ms  providers={sess.get_providers()}  output={out[0].shape}")


Actual test

In [6]:
bench("CPU", ["CPUExecutionProvider"])
bench("CUDA", ["CUDAExecutionProvider", "CPUExecutionProvider"])
bench("TensorRT+CUDA", ["TensorrtExecutionProvider", "CUDAExecutionProvider", "CPUExecutionProvider"])


CPU               30.33 ms  providers=['CPUExecutionProvider']  output=(1, 1000)
CUDA               1.80 ms  providers=['CUDAExecutionProvider', 'CPUExecutionProvider']  output=(1, 1000)
TensorRT+CUDA      1.01 ms  providers=['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']  output=(1, 1000)
